In [2]:
#import streamlit as st
import pandas as pd
import numpy as np



# ── Position group ────────────────────────────────────────────────────────────
def pos_group(p):
    if pd.isna(p):
        return np.nan
    p = int(p)
    if p <= 2:  return 'opener'
    if p <= 5: return 'upper middle'
    if p <= 7:  return 'lower middle'
    return 'tail'  # 8-11

# ── Load & enrich (shared with sguru.py) ─────────────────────────────────────

def load_and_enrich(path: str, block_size: int = 8, k: float = 0.2) -> pd.DataFrame:
    df = pd.read_csv(path, low_memory=False)
    df.loc[df.bat=='Yousuf Youhana','bat'] = 'Mohammad Yousuf'

    for col in ['year', 'runs', 'balls', 'strike_rate', 'batting_position', 'inns', 'is_out']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    seasons_ordered = sorted(df['season'].dropna().unique())
    season_rank = {s: i for i, s in enumerate(seasons_ordered)}
    df['season_rank'] = df['season'].map(season_rank)
    df['era_block'] = (df['season_rank'] // block_size) * block_size
    df['pos_group'] = df['batting_position'].apply(pos_group)
    

    # ── Home / Away ──
    NEUTRAL_MAP = {'United Arab Emirates': 'Pakistan'}
    def get_home_away(row):
        home_country = NEUTRAL_MAP.get(row['country'], row['country'])
        return 'Home' if row['team_bat'] == home_country else 'Away'
    df['home_away'] = df.apply(get_home_away, axis=1)

    top7 = df[df['batting_position'] <= 5]

    # ── Helper: compute z-score weights and adj_runs for any baseline ──
    def z_adjust(df_full, base_col):
        mu  = df_full[base_col].mean()
        std = df_full[base_col].std()
        df_full['_z'] = (df_full[base_col] - mu) / std
        df_full['_w'] = np.exp(-k * df_full['_z'])
        adj = df_full['runs'] * df_full['_w']
        df_full.drop(columns=['_z', '_w'], inplace=True)
        return adj

    # ── Baseline 1: ERA ──
    era = (
        top7.groupby('era_block')
        .apply(lambda g: g['runs'].sum() / g['is_out'].sum())
        .reset_index(name='base_era')
    )
    df = df.merge(era, on='era_block', how='left')
    df['adj_runs_era'] = z_adjust(df, 'base_era')

    # ── Baseline 2: ERA-COUNTRY ──
    era_ctry = (
        top7.groupby(['era_block', 'country','team_bowl'])
        .apply(lambda g: g['runs'].sum() / g['is_out'].sum())
        .reset_index(name='base_era_country')
    )
    df = df.merge(era_ctry, on=['era_block', 'country','team_bowl'], how='left')
    #df['adj_runs_era_country'] = z_adjust(df, 'base_era_country')

    # ── Baseline 3: ERA-POSITION ──
    era_pos = (
        top7.groupby(['era_block', 'pos_group'])
        .apply(lambda g: g['runs'].sum() / g['is_out'].sum())
        .reset_index(name='base_era_pos')
    )
    df = df.merge(era_pos, on=['era_block', 'pos_group'], how='left')

    # Z-score within each pos_group separately
    pos_stats = (
        df.groupby('pos_group')['base_era_pos']
        .agg(mu='mean', std='std')
        .reset_index()
    )
    df = df.merge(pos_stats, on='pos_group', how='left')
    df['_z_pos'] = (df['base_era_pos'] - df['mu']) / df['std']
    df['adj_runs_era_pos'] = df['runs'] * np.exp(-k * df['_z_pos'])
    df.drop(columns=['_z_pos', 'mu', 'std'], inplace=True)

    # ── Baseline 4: ERA-OPPOSITION ──
    era_opp = (
        top7.groupby(['era_block', 'team_bowl'])
        .apply(lambda g: g['runs'].sum() / g['is_out'].sum())
        .reset_index(name='base_era_opp')
    )
    df = df.merge(era_opp, on=['era_block', 'team_bowl'], how='left')
    df['adj_runs_era_opp'] = z_adjust(df, 'base_era_opp')

    # ── % of team runs ──
    if 'team_innings_runs' in df.columns:
        df['team_runs_pct'] = df['runs'] / df['team_innings_runs'] * 100

    # ── Match factor ──
    # ── Match factor ──
    top8 = df[df['batting_position'] < 12][['p_match', 'runs', 'is_out']].copy() # top8 runs is now all runs: May 1 2026
    match_agg = (
        top8.groupby('p_match')
        .agg(top8_runs=('runs', 'sum'), top8_outs=('is_out', 'sum'))
        .reset_index()
    )
    match_agg['match_ave'] = match_agg['top8_runs'] / match_agg['top8_outs'].replace(0, np.nan)
    df = df.merge(match_agg[['p_match', 'match_ave']], on='p_match', how='left')
    df['match_factor'] = df['runs'] / df['match_ave']
    df.drop(columns=['match_ave'], inplace=True)

    return df

/Users/hganjoo/miniconda3/envs/sle/lib/python3.9/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [3]:
df_raw = load_and_enrich('test_batting_innings.csv')

/var/folders/hg/4pj6768x3q39vm215pfl2bcm0000gn/T/ipykernel_20238/3692122383.py:55: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  top7.groupby('era_block')
/var/folders/hg/4pj6768x3q39vm215pfl2bcm0000gn/T/ipykernel_20238/3692122383.py:64: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  top7.groupby(['era_block', 'country','team_bowl'])
/var/folders/hg/4pj6768x3q39vm215pfl2bcm0000gn/T/ipykernel_20238/3692122383.py:73: Fu

In [4]:
t = df_raw[df_raw.bat=='RT Ponting']

In [19]:
data = (
    t.groupby(['era_block', 'team_bowl'])
    .agg(
        base_era_country=('base_era_opp', 'mean'),
        runs=('runs', 'sum'),
        is_out=('is_out', 'sum'),
        season_min=('season', 'min'),
        season_max=('season', 'max'),
    )
    .reset_index()
)
data.sort_values('base_era_country',ascending=False)

,era_block,team_bowl,base_era_country,runs,is_out,season_min,season_max
11,176,Bangladesh,59.535135,69.0,2,2003,2003
20,184,Bangladesh,59.367647,191.0,2,2005/06,2005/06
19,176,Zimbabwe,54.073298,259.0,2,2003/04,2003/04
27,184,West Indies,47.629393,652.0,10,2005/06,2008
33,192,Sri Lanka,46.050505,124.0,4,2011,2011
18,176,West Indies,45.276074,523.0,4,2003,2003
30,192,New Zealand,45.155738,168.0,6,2009/10,2011/12
29,192,India,44.949296,768.0,9,2010/11,2011/12
1,160,Sri Lanka,44.932642,193.0,4,1995/96,1995/96
14,176,New Zealand,43.534091,689.0,8,2001/02,2004/05


In [20]:
t.base_era_opp.mean()

40.294115620830425

In [21]:
(t.runs.sum() / t.is_out.sum()) / t.base_era_opp.mean()

1.2868557202304949